# 2b — The Cost Function Surface

The brute-force matcher in `synthetic_scan_matching_step_1.py` tries every
translation on a grid and keeps the one with the lowest error. This notebook
looks at that error as a *function* of the candidate transform, so we can
see the landscape the search is exploring.

**Learning goals**

1. Define the cost of an alignment as a sum (or mean) of squared errors.
2. Plot the cost over a grid of `(dx, dy)` and see the "bowl".
3. See what happens to the bowl when the scene has no distinguishing features.
4. Extend the picture to rotation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Two synthetic scans

We reuse the setup from the step-1 script: a scan at time 0, and the same
scan after the robot has moved by `(true_dx, true_dy)`. Because the data is
synthetic the $i$-th point of one scan corresponds to the $i$-th point of
the other, so no nearest-neighbour search is needed yet.

In [ ]:
def make_corner(num_points=40, length=4.0):
    """An L-shaped corner: one wall along +x, one along +y."""
    s = np.linspace(0, length, num_points)
    return np.vstack((np.column_stack((s, np.zeros_like(s))),
                      np.column_stack((np.zeros_like(s), s))))

def apply_movement(scan, dx, dy):
    """A robot motion of (+dx, +dy) makes the scene appear to move by (-dx, -dy)."""
    return scan - np.array([dx, dy])

true_dx, true_dy = 0.6, 0.3
scan_t0 = make_corner()
scan_t1 = apply_movement(scan_t0, true_dx, true_dy)

## Defining the cost

For a candidate correction $(dx, dy)$ applied to `scan_t1`, the residual of
point $i$ is the vector from the shifted point to its partner in `scan_t0`.
The **sum of squared errors** is

$$\mathrm{SSE}(dx, dy) = \sum_i \big\| \mathbf{p}_i^{(0)} - (\mathbf{p}_i^{(1)} + (dx, dy)) \big\|^2 .$$

Dividing by the number of points gives the **mean squared error**, which
has the advantage of not growing with scan size — handy later when we
compare scans with different numbers of points, and when choosing a
gradient-descent step size.

In [ ]:
def sse(reference, scan, dx, dy):
    corrected = scan + np.array([dx, dy])
    return np.sum((reference - corrected) ** 2)

def mse(reference, scan, dx, dy):
    return sse(reference, scan, dx, dy) / len(reference)

print("cost at (0, 0):        ", round(sse(scan_t0, scan_t1, 0, 0), 3))
print("cost at the true shift:", round(sse(scan_t0, scan_t1, true_dx, true_dy), 3))

## The cost surface over translation

Evaluate the cost on a grid of candidate `(dx, dy)` and plot it two ways: a
3D surface and a contour map. This grid is exactly what the brute-force
search walks through.

In [ ]:
def cost_grid(reference, scan, search_range=1.5, step=0.05, cost=mse):
    candidates = np.arange(-search_range, search_range + step, step)
    DX, DY = np.meshgrid(candidates, candidates)
    C = np.array([[cost(reference, scan, dx, dy) for dx in candidates] for dy in candidates])
    return DX, DY, C

DX, DY, C = cost_grid(scan_t0, scan_t1)

fig = plt.figure(figsize=(13, 5))
ax3d = fig.add_subplot(1, 2, 1, projection="3d")
ax3d.plot_surface(DX, DY, C, cmap="viridis", alpha=0.9)
ax3d.set_xlabel("dx"); ax3d.set_ylabel("dy"); ax3d.set_zlabel("MSE")
ax3d.set_title("Cost surface: a bowl")

ax2d = fig.add_subplot(1, 2, 2)
cs = ax2d.contour(DX, DY, C, levels=25, cmap="viridis")
ax2d.clabel(cs, inline=True, fontsize=7)
ax2d.plot(true_dx, true_dy, "r*", ms=14, label="true motion")
i, j = np.unravel_index(np.argmin(C), C.shape)
ax2d.plot(DX[i, j], DY[i, j], "wo", mec="k", ms=8, label="grid minimum")
ax2d.set_xlabel("dx"); ax2d.set_ylabel("dy"); ax2d.axis("equal"); ax2d.legend()
ax2d.set_title("Same surface as contours")
plt.show()

print(f"grid minimum at dx={DX[i, j]:.2f}, dy={DY[i, j]:.2f}")

Because the cost is a sum of squares of something *linear* in `(dx, dy)`,
the surface is a perfect quadratic bowl with a single minimum at the true
motion. That is the ideal case, and it is what makes the optimisation
methods in Module 3 work so well.

## When the bowl becomes a valley

Now repeat the experiment with a **single straight wall** (the step-1
scene) instead of a corner — but this time pretend the sensor gives us no
point-to-point correspondence and let each point match whatever point on
the wall is nearest to it. That is what a real matcher has to do.

In [ ]:
def make_wall(num_points=50, length=10.0):
    x = np.linspace(-length / 2, length / 2, num_points)
    return np.column_stack((x, np.full(num_points, 5.0)))

def nearest_neighbour_mse(reference, scan, dx, dy):
    """Each corrected point is compared with its *closest* reference point."""
    corrected = scan + np.array([dx, dy])
    d2 = ((corrected[:, None, :] - reference[None, :, :]) ** 2).sum(axis=2)
    return d2.min(axis=1).mean()

wall_t0 = make_wall()
wall_t1 = apply_movement(wall_t0, true_dx, true_dy)

DXw, DYw, Cw = cost_grid(wall_t0, wall_t1, cost=nearest_neighbour_mse)

fig = plt.figure(figsize=(13, 5))
ax3d = fig.add_subplot(1, 2, 1, projection="3d")
ax3d.plot_surface(DXw, DYw, Cw, cmap="viridis", alpha=0.9)
ax3d.set_xlabel("dx"); ax3d.set_ylabel("dy"); ax3d.set_zlabel("NN-MSE")
ax3d.set_title("A flat wall: the bowl becomes a valley")

ax2d = fig.add_subplot(1, 2, 2)
ax2d.contourf(DXw, DYw, Cw, levels=30, cmap="viridis")
ax2d.plot(true_dx, true_dy, "r*", ms=14, label="true motion")
ax2d.set_xlabel("dx"); ax2d.set_ylabel("dy"); ax2d.axis("equal"); ax2d.legend()
plt.show()

Sliding the scan along the wall (changing `dx`) barely changes the cost:
the floor of the valley is nearly flat in that direction, so *any* `dx`
looks about as good as the true one. Only `dy` — the distance to the wall
— is well determined. This is the **corridor problem**, revisited in
[5a](5a_the_corridor_problem.ipynb).

## Adding rotation

A real motion has three degrees of freedom: `(dx, dy, θ)`. We can't draw a
4D surface, so take slices. Below, `dy` is fixed at its true value and the
cost is plotted over `(dx, θ)` for the corner scene.

In [ ]:
def rotation_matrix(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

def sse_3dof(reference, scan, dx, dy, theta):
    corrected = scan @ rotation_matrix(theta).T + np.array([dx, dy])
    return np.sum((reference - corrected) ** 2)

true_theta = np.radians(10)
# Robot turned by +10° and moved (0.6, 0.3): the scene rotates by -10° and shifts back.
scan_t1_rot = (scan_t0 - np.array([true_dx, true_dy])) @ rotation_matrix(-true_theta).T

dxs = np.arange(-1.5, 1.5, 0.05)
thetas = np.radians(np.arange(-45, 45, 1.0))
DXr, TH = np.meshgrid(dxs, thetas)
Cr = np.array([[sse_3dof(scan_t0, scan_t1_rot, dx, true_dy, th) for dx in dxs] for th in thetas])

fig, ax = plt.subplots(figsize=(7, 5))
cf = ax.contourf(DXr, np.degrees(TH), Cr, levels=30, cmap="viridis")
fig.colorbar(cf, label="SSE")
ax.plot(true_dx, np.degrees(true_theta), "r*", ms=14, label="true (dx, θ)")
ax.set_xlabel("dx (m)"); ax.set_ylabel("θ (degrees)"); ax.legend()
ax.set_title("Cost over (dx, θ) with dy fixed at its true value")
plt.show()

Rotation adds a whole new axis to search. A brute-force grid of 60 × 60
translations already costs 3 600 evaluations; adding 90 rotation steps makes
it 324 000. That growth is the *curse of dimensionality* mentioned in the
curriculum, and the reason Module 3 replaces the grid search with an
optimiser that follows the slope of the bowl downhill.

## Summary

* The cost of an alignment is the (mean) squared distance between matched points.
* With good correspondences and a feature-rich scene the cost is a clean
  bowl with one minimum at the true motion.
* A featureless wall flattens the bowl into a valley: motion along the wall is unobservable.
* Each extra degree of freedom multiplies the size of a brute-force search.

**Next:** [3a — Gradient descent](3a_gradient_descent_toy.ipynb) walks down the bowl instead of sampling it.